# BioPred Phase 2 — Selected Model Validation and Leakage Audit

Notebook 10 is a formal pre-test validation gate for the BioPred selected model.

Notebook 09 selected `extra_trees` as the current CV-stage model under scaffold-aware training cross-validation. The final scaffold-held-out test set was not used during model benchmarking or model-family selection.

The purpose of this notebook is to determine whether the selected ExtraTrees CV result is credible, non-leaky, and robust enough to justify moving to a one-time final scaffold-held-out test evaluation in Notebook 11.

This notebook does **not** tune the selected model, prune features, compute final test metrics, perform threshold optimization, or produce ranked-output decisioning artifacts.

The final scaffold-held-out test labels remain locked until this audit passes.

## Objectives

The objective of this notebook is to decide whether the selected ExtraTrees CV result from Notebook 09 is credible enough to justify a one-time final scaffold-held-out test evaluation in Notebook 11.

Notebook 10 will support that decision by verifying:

1. **Selection provenance**
   - Notebook 09 selected `extra_trees`.
   - Selection occurred under scaffold-aware training cross-validation policy.
   - Final scaffold-held-out test status is `not_used`.

2. **Artifact integrity**
   - Locked Notebook 07 train/test artifacts load correctly.
   - Train/test dimensions match the locked split.
   - X, y, metadata, and split-assignment artifacts remain row-aligned.
   - Train/test row identities remain disjoint.

3. **Feature-policy compliance**
   - X matrices do not contain target labels.
   - X matrices do not contain potency summaries used to define labels.
   - X matrices do not contain evidence/diagnostic metadata.
   - X matrices do not contain molecule identifiers, SMILES, InChIKey, or scaffold labels.
   - Train/test feature columns match the expected RDKit + Morgan feature structure.

4. **Scaffold split integrity**
   - Train/test scaffold overlap remains zero.
   - Scaffold group assignments are intact.

5. **Duplicate and near-duplicate risk**
   - Exact molecule duplication across train/test is checked using available identifiers.
   - Test-to-train nearest-neighbor Morgan Tanimoto support is summarized for later interpretation of final test results.

6. **CV fold-level robustness**
   - Saved Notebook 09 fold-level ExtraTrees results are reviewed.
   - The audit checks whether strong mean CV performance was supported across folds or driven by one unusually favorable fold.
   - Weak-fold AP and early-retrieval behavior are reviewed before final test evaluation.

7. **Label-permutation negative control**
   - Activity labels are shuffled within the training set.
   - The selected ExtraTrees workflow is rerun under the same scaffold-aware CV structure.
   - The audit checks whether shuffled-label performance collapses relative to the real-label benchmark.
   - This tests whether the strong CV result is likely driven by real structure-activity signal rather than metric artifacts, split artifacts, or model-capacity illusion.

8. **Final audit decision**
   - `pass`: proceed to Notebook 11 final scaffold-held-out test evaluation.
   - `conditional_pass`: resolve specific concerns before final test evaluation.
   - `fail`: do not touch final test metrics until the issue is corrected.

In [1]:
# Force notebook runtime to project root
%cd /home/ryanm/projects/BioPred

from pathlib import Path
import sys
import json
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
)

# define paths and link src.paths
SRC_DIR = Path.cwd() / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from src.biopred import paths

from biopred.metrics import (
    precision_at_k_percent,
    enrichment_factor_at_k_percent
)


/home/ryanm/projects/BioPred


In [2]:
# load Notebook 09 benchmark config and verify the selected-model contract
# note this is just a provenance check, not an analysis.

BENCHMARK_DIR = paths.MODELING_DIR / "model_benchmarking"
CONFIG_PATH = BENCHMARK_DIR / "model_benchmark_config.json"

assert CONFIG_PATH.exists(), f"Missing Notebook 09 config: {CONFIG_PATH}"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    benchmark_config = json.load(f)

expected_contract = {
    "selected_cv_stage_model_name" : "extra_trees",
    "primary_challenger_model_name" : "extra_trees",
    "final_test_status" : "not_used",
    "selected_split_seed" : 33,
    "target_column" : "active_median_px_6_0",
    "scaffold_column" : "bemis_murcko_scaffold",
    "challenger_replaces_incumbent" : True, 
}

accepted_selection_scopes = {
    "scaffold_aware_training_policy",
}

for key, expected_value in expected_contract.items():
    observed_value = benchmark_config.get(key)
    assert observed_value == expected_value, (
        f"Notebook 09 contract mismatch for {key!r}: "
        f"expected {expected_value!r}, observed {observed_value!r}"
    )

selection_scope = benchmark_config.get("selection_scope")
assert selection_scope in accepted_selection_scopes, (
    f"Unexpected selection scope: {selection_scope!r}"
)

print("Notebook 09 selection contract passed.")
print(f"Selected model: {benchmark_config['selected_cv_stage_model_name']}")
print(f"Selection scope: {selection_scope}")
print(f"Final test status: {benchmark_config['final_test_status']}")

Notebook 09 selection contract passed.
Selected model: extra_trees
Selection scope: scaffold_aware_training_policy
Final test status: not_used


### **Section 2: Load Locked Notebook 07 Modeling Artifacts**

This section identifies the frozen train/test modeling artifacts created by Notebook 07.

The final scaffold-held-out test set is referenced only for structural validation: artifact existence, row counts, index alignment, partition separation, scaffold integrity, and leakage-risk auditing.

This section does **not** fit models, generate predictions, compute final test metrics, rank test molecules, or use test labels for model selection.

In [3]:
# load all of the Notebook 07 modeling artifacts
# will check shape to verify

MODELING_DIR = paths.MODELING_DIR

X_train_raw = pd.read_parquet(MODELING_DIR / "X_train_raw.parquet")
X_test_raw = pd.read_parquet(MODELING_DIR / "X_test_raw.parquet")

y_train = pd.read_parquet(MODELING_DIR / "y_train.parquet")
y_test = pd.read_parquet(MODELING_DIR / "y_test.parquet")

metadata_train = pd.read_parquet(MODELING_DIR / "metadata_train.parquet")
metadata_test = pd.read_parquet(MODELING_DIR / "metadata_test.parquet")

split_assignments_train = pd.read_parquet(MODELING_DIR / "split_assignments_train.parquet")
split_assignments_test = pd.read_parquet(MODELING_DIR / "split_assignments_test.parquet")

shape_summary = pd.DataFrame(
    [
        {"artifact" : "X_train_raw", "rows" : X_train_raw.shape[0], "columns" : X_train_raw.shape[1]},
        {"artifact" : "X_test_raw", "rows" : X_test_raw.shape[0], "columns" : X_test_raw.shape[1]},
        {"artifact" : "y_train", "rows" : y_train.shape[0], "columns" : y_train.shape[1]},
        {"artifact" : "y_test", "rows" : y_test.shape[0], "columns" : y_test.shape[1]},
        {"artifact" : "metadata_train", "rows" : metadata_train.shape[0], "columns" : metadata_train.shape[1]},
        {"artifact" : "metadata_test",  "rows" : metadata_test.shape[0], "columns" : metadata_test.shape[1]},
        {"artifact" : "split_assignments_train", "rows" : split_assignments_train.shape[0], "columns" : split_assignments_test.shape[1]},
        {"artifact" : "split_assignments_test", "rows" : split_assignments_test.shape[0], "columns" : split_assignments_test.shape[1]},
    ]
)

shape_summary



,artifact,rows,columns
0,X_train_raw,1236,2265
1,X_test_raw,310,2265
2,y_train,1236,1
3,y_test,310,1
4,metadata_train,1236,12
5,metadata_test,310,12
6,split_assignments_train,1236,3
7,split_assignments_test,310,3


In [4]:
# check artifact alignment and partition separation
# make sure that X, y, metadata, and split assignments are and remain row-aligned
# within train/test and that train/test rows are disjoint.

alignment_checks = {
    # train partition alignment
    "train_X_y_index_aligned" : X_train_raw.index.equals(y_train.index),
    "train_X_metadata_index_aligned" : X_train_raw.index.equals(metadata_train.index),
    "train_X_split_index_aligned" : X_train_raw.index.equals(split_assignments_train.index),

    # test partition alignment
    "test_X_y_index_aligned" : X_test_raw.index.equals(y_test.index),
    "test_X_metadata_index_aligned" : X_test_raw.index.equals(metadata_test.index),
    "test_X_split_index_aligned" : X_test_raw.index.equals(split_assignments_test.index),

    # no duplicate row indices in core artifacts
    "train_index_unique" : not X_train_raw.index.has_duplicates,
    "test_index_unique" : not X_test_raw.index.has_duplicates,

    # make sure train/test are row-separated
    "train_test_index_disjoint" : len(X_train_raw.index.intersection(X_test_raw.index)) == 0,
}

alignment_summary = pd.DataFrame(
    [{"check" : k, "passed" : v} for k, v in alignment_checks.items()]
)


# quick assertion
assert alignment_summary["passed"].all(), "One or more artifact alignment checks failed."

alignment_summary


,check,passed
0,train_X_y_index_aligned,True
1,train_X_metadata_index_aligned,True
2,train_X_split_index_aligned,True
3,test_X_y_index_aligned,True
4,test_X_metadata_index_aligned,True
5,test_X_split_index_aligned,True
6,train_index_unique,True
7,test_index_unique,True
8,train_test_index_disjoint,True


### **Section 3: Feature-Policy Compliance**

This section audits whether the selected-model feature matrices comply with the BioPred feature policy before final test evaluation.

The audit checks that `X_train_raw` and `X_test_raw` do not contain forbidden leakage columns:

- target-derived labels;
- potency summaries used to define labels;
- assay/evidence diagnostic metadata;
- molecule identifiers;
- structure strings;
- scaffold labels.

The expected result is that no forbidden columns are present in either feature matrix.

This section does not tune, prune, refit, score, or evaluate the selected model.

In [5]:
# feature-policy compliance
# verify that selected-model feature matrices do not contain forbidden leakage columns.

target_columns = {
    "active_median_px_5_5",
    "active_median_px_6_0",
    "active_median_px_6_5",
}

potency_summary_columns = {
    "mean_px",
    "median_px",
    "min_px",
    "max_px",
    "px_std",
    "px_range",
}

diagnostic_evidence_columns = {
    "n_activity_records",
    "n_assays",
    "n_documents",
    "n_targets",
    "n_endpoint_types",
    "has_record_label_conflict_5_5",
    "has_record_label_conflict_6_0",
    "has_record_label_conflict_6_5",
}

identifier_structure_columns = {
    "molregno",
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_inchi_key",
    "bemis_murcko_scaffold",
}

forbidden_columns = (
    target_columns
    | potency_summary_columns
    | diagnostic_evidence_columns
    | identifier_structure_columns
)

# find any potential forbidden cols present in train/test
train_forbidden = sorted(set(X_train_raw.columns) & forbidden_columns)
test_forbidden = sorted(set(X_test_raw.columns) & forbidden_columns)

leakage_audit = pd.DataFrame(
    [
        {
            "partition" : "train",
            "n_forbidden_columns" : len(train_forbidden),
            "forbidden_columns" : train_forbidden,
        },
        {
            "partition" : "test",
            "n_forbidden_columns" : len(test_forbidden),
            "forbidden_columns" : test_forbidden,
        },
   ]
)

# quick assertion to verify
assert leakage_audit["n_forbidden_columns"].eq(0).all(), (
    "Forbidden leakage columns found in selected-model feature matrices."
)


leakage_audit



,partition,n_forbidden_columns,forbidden_columns
0,train,0,[]
1,test,0,[]



No forbidden leakage columns were found in `X_train_raw` or `X_test_raw`.

This supports the claim that the Notebook 09 ExtraTrees benchmark used model inputs consistent with the BioPred feature policy: structure-derived features only, with target labels, potency summaries, evidence diagnostics, identifiers, structure strings, and scaffold labels excluded from the feature matrices.

In [6]:
# feature structure confirmation
# confirm selected-model feature matrices contain the expected allowed feature families

EXPECTED_RDKIT_FEATURES = 217
EXPECTED_MORGAN_FEATURES = 2048

# identify feature-family columns by prefix
train_rdkit_cols = [x for x in X_train_raw.columns if x.startswith("rdkit_")]
train_morgan_cols = [x for x in X_train_raw.columns if x.startswith("morgan_")]

test_rdkit_cols = [x for x in X_test_raw.columns if x.startswith("rdkit_")]
test_morgan_cols = [x for x in X_test_raw.columns if x.startswith("morgan_")]

feature_structure_audit = pd.DataFrame(
    [
        {
            "check" : "train_rdkit_feature_count",
            "observed" : len(train_rdkit_cols),
            "expected" : EXPECTED_RDKIT_FEATURES,
            "passed" : len(train_rdkit_cols) == EXPECTED_RDKIT_FEATURES,
        },
        {
            "check" : "train_morgan_feature_count",
            "observed" : len(train_morgan_cols),
            "expected" : EXPECTED_MORGAN_FEATURES,
            "passed" : len(train_morgan_cols) == EXPECTED_MORGAN_FEATURES,
        },
        {
            "check" : "test_rdkit_feature_count",
            "observed" : len(test_rdkit_cols),
            "expected" : EXPECTED_RDKIT_FEATURES,
            "passed" : len(test_rdkit_cols) == EXPECTED_RDKIT_FEATURES,
        },
        {
            "check" : "test_morgan_feature_count",
            "observed" : len(test_morgan_cols),
            "expected" : EXPECTED_MORGAN_FEATURES,
            "passed" : len(test_morgan_cols) == EXPECTED_MORGAN_FEATURES,
        },
        {
            "check" : "train_test_feature_columns_identical",
            "observed" : X_train_raw.columns.equals(X_test_raw.columns),
            "expected" : True,
            "passed" : X_train_raw.columns.equals(X_test_raw.columns),
        },
    ]
)

# a quick assertion to verify
assert feature_structure_audit["passed"].all(), (
    "Selected-model feature structure audit failed."
)

feature_structure_audit

,check,observed,expected,passed
0,train_rdkit_feature_count,217,217,True
1,train_morgan_feature_count,2048,2048,True
2,test_rdkit_feature_count,217,217,True
3,test_morgan_feature_count,2048,2048,True
4,train_test_feature_columns_identical,True,True,True


### **Section 4: Scaffold Split Integrity**

This section verifies that the locked Notebook 07 scaffold split remains intact before final test evaluation.

The audit checks that:

- train/test scaffold overlap remains zero;
- scaffold group assignments are present and non-missing;
- scaffold concentration is reviewed only to assess whether CV performance could be dominated by a small number of large scaffolds.

This section does not compute final test model performance.

In [7]:
# scaffold split integrity
# confirm train/test scaffold separation remains intact before final test evaluation.

SCAFFOLD_COL = "bemis_murcko_scaffold"

train_scaffolds = split_assignments_train[SCAFFOLD_COL]
test_scaffolds = split_assignments_test[SCAFFOLD_COL]

# build scaffold sets
train_scaffold_set = set(train_scaffolds)
test_scaffold_set = set(test_scaffolds)

# compute the train/test scaffold overlap.
overlapping_scaffolds = sorted(train_scaffold_set & test_scaffold_set)

scaffold_integrity_checks = {
    "scaffold_col_in_train_split_assignments" : SCAFFOLD_COL in split_assignments_train.columns,
    "scaffold_col_in_test_split_assignments" : SCAFFOLD_COL in split_assignments_test.columns,
    "train_scaffolds_non_missing" : train_scaffolds.notna().all(),
    "test_scaffolds_non_missing" : test_scaffolds.notna().all(),
    "train_test_scaffold_overlap_zero" : len(overlapping_scaffolds) == 0,
}

scaffold_integrity_summary = pd.DataFrame(
    [{"check" : k, "passed" : v} for k, v in scaffold_integrity_checks.items()]
)

scaffold_integrity_summary

,check,passed
0,scaffold_col_in_train_split_assignments,True
1,scaffold_col_in_test_split_assignments,True
2,train_scaffolds_non_missing,True
3,test_scaffolds_non_missing,True
4,train_test_scaffold_overlap_zero,True



The scaffold column is present in both train and test split-assignment artifacts, scaffold labels are non-missing, and train/test scaffold overlap is zero.

This confirms that the locked Notebook 07 scaffold-held-out train/test separation remains intact before final test evaluation.

Scaffold concentration was already reviewed during split construction in Notebook 07. Notebook 10 does not recompute that analysis because the purpose here is to verify that the scaffold split remains intact, not to repeat pre-modeling split diagnostics.

### **Section 5: Duplicate and Near-Duplicate Risk**

This section audits whether the selected ExtraTrees CV result could be inflated by molecule-level redundancy across the locked train/test split.

The scaffold split has already been confirmed as scaffold-disjoint, but scaffold-disjoint does not necessarily mean chemistry-disjoint. Molecules from different Bemis-Murcko scaffolds can still share high fingerprint similarity, and exact duplicate structures may still indicate artifact or aggregation problems if present across partitions.

This section checks two related risks:

1. **Exact duplicate risk**
   - duplicated SMILES, InChIKey, or molecule identifiers across train/test where available;
   - this would represent a serious split-integrity concern.

2. **Near-duplicate / chemical-neighborhood risk**
   - nearest-neighbor Tanimoto similarity from each test molecule to the training set;
   - this does not automatically invalidate the split, but it helps interpret final test performance.

The goal is not to remove molecules, tune the model, or alter the final test set. The goal is to understand whether future final-test performance should be interpreted as scaffold-generalization, close-analog retrieval, or potentially contaminated evaluation.

This section does not compute model predictions or final test metrics.

In [8]:
# perform exact duplicate risk audit
# check whether exact molecule identities appear in both train and test partitions.

identifier_cols_to_check = [
    "molregno",
    "molecule_chembl_id",
    "canonical_smiles",
    "standard_inchi_key",
]

exact_duplicate_checks = []

for col in identifier_cols_to_check:
    # skip cols that aren't in both train and test
    if col in metadata_train.columns and col in metadata_test.columns:
        train_values = set(metadata_train[col].dropna())
        test_values = set(metadata_test[col].dropna())

        overlapping_values = train_values & test_values

        exact_duplicate_checks.append(
            {
                "identifier_column" : col,
                "train_unique_value" : len(train_values),
                "test_unique_values" : len(test_values),
                "n_cross_partition_overlaps" : len(overlapping_values),
                "passed" : len(overlapping_values) == 0,
                "example_overlaps" : sorted(list(overlapping_values))[:5],
            }
        )

exact_duplicate_audit = pd.DataFrame(exact_duplicate_checks)

# quick assertion to verify
assert exact_duplicate_audit["passed"].all(), (
    "Exact molecule identity overlap found across train/test partitions."
)

exact_duplicate_audit

,identifier_column,train_unique_value,test_unique_values,n_cross_partition_overlaps,passed,example_overlaps
0,molregno,1236,310,0,True,[]
1,molecule_chembl_id,1236,310,0,True,[]
2,canonical_smiles,1236,310,0,True,[]
3,standard_inchi_key,1236,310,0,True,[]


**Nearest-Neighbor Tanimoto Support**

This section summarizes how chemically close each final-test molecule is to its nearest training molecule using Morgan fingerprint Tanimoto similarity.

This check does not compute model predictions or final test performance. It is a support/interpretability audit.

High nearest-train Tanimoto similarity does not automatically indicate leakage. In medicinal chemistry datasets, close analogs across scaffold-disjoint partitions can be legitimate. However, if many final-test molecules have very high nearest-train similarity, then strong final-test performance should be interpreted more cautiously as close-analog retrieval rather than broad chemical generalization.

The output of this section will be used later to contextualize Notebook 11 final test results.

In [9]:
# prepare Morgan fingerprint matrices for Tanimoto support audit.
# extract binary Morgan fingerprint columns from train/test feature matrices

morgan_cols = train_morgan_cols.copy()

morgan_column_checks = {
    "n_morgan_cols_expected" : len(morgan_cols) == 2048,
    "morgan_cols_match_test" : morgan_cols == list(X_test_raw.columns[X_test_raw.columns.str.startswith("morgan_")])
}

morgan_column_audit = pd.DataFrame(
    [{"check" : k, "passed" : v} for k, v in morgan_column_checks.items()]
)

# quick assertion to verify
assert morgan_column_audit["passed"].all(), (
    "Morgan fingerprint column audit failed."
)

morgan_column_audit

,check,passed
0,n_morgan_cols_expected,True
1,morgan_cols_match_test,True


A note on our Tanimoto Similarity usage:

Tanimoto similarity is computed directly from the saved binary Morgan fingerprint columns used as model inputs. This avoids regenerating fingerprints from SMILES and ensures the support audit reflects the exact feature representation available to the selected model.

In [10]:
# nearest-neighbor Tanimoto support audit.
# compute each test molecule's nearest Morgan-fingerprint neighbor in the training set.

# extract Morgan fingerprint matrices as integer/boolean numpy arrays.
X_train_morgan = X_train_raw[morgan_cols].to_numpy(dtype=np.int32)
X_test_morgan = X_test_raw[morgan_cols].to_numpy(dtype=np.int32)

# compute bit counts for train and test molecules
train_bit_counts = X_train_morgan.sum(axis = 1)
test_bit_counts = X_test_morgan.sum(axis = 1)

nearest_train_tanimoto = []
nearest_train_pos = []

# loop over test molecules
for test_fp in X_test_morgan:
    # compute intersection count with every training fingerprint
    intersections = X_train_morgan @ test_fp

    # compute union count for every training fingerprint.
    unions = train_bit_counts + test_fp.sum() - intersections

    # compute Tanimoto safely
    similarities = np.divide(
        intersections,
        unions,
        out = np.zeros_like(intersections, dtype=float),
        where = unions != 0,
    )

    # store nearest similarity and nearest training row position.
    nearest_train_tanimoto.append(similarities.max())
    nearest_train_pos.append(similarities.argmax())

nearest_train_tanimoto = np.array(nearest_train_tanimoto)
nearest_train_pos = np.array(nearest_train_pos)

tanimoto_support = pd.DataFrame(
    {
        "nearest_train_tanimoto" : nearest_train_tanimoto,
        "nearest_train_position" : nearest_train_pos,
    },
    index = X_test_raw.index,
)

tanimoto_support.head()


,nearest_train_tanimoto,nearest_train_position
24,0.836364,542
34,0.250000,446
44,0.283333,850
45,0.466667,185
49,0.280000,407


The Tanimoto is working, now let's look at a distribution of those values.

In [11]:
# summarize nearest-train Tanimoto support.

nn_tanimoto = tanimoto_support["nearest_train_tanimoto"]

tanimoto_summary = nn_tanimoto.describe(
    percentiles=[0.1, 0.25, 0.50, 0.75, 0.90, 0.95]
)

tanimoto_summary

count    310.000000
mean       0.635863
std        0.179930
min        0.172414
10%        0.307535
25%        0.567530
50%        0.691608
75%        0.754035
90%        0.827250
95%        0.872553
max        0.945455
Name: nearest_train_tanimoto, dtype: float64

In [12]:
# define support bands
support_bins = [0.0, 0.50, 0.70, 0.85, 1.01]

support_labels = [
    "low_support_<0.50",
    "moderate_support_0.50_0.70",
    "high_support_0.70_0.85",
    "very_high_support_>=0.85",
]

tanimoto_support["support_band"] = pd.cut(
    tanimoto_support["nearest_train_tanimoto"],
    bins = support_bins,
    labels = support_labels,
    include_lowest= True,
    right = False,
)

support_band_summary = (
    tanimoto_support["support_band"]
    .value_counts(sort=False)
    .rename_axis("support_band")
    .reset_index(name = "n_test_molecules")
)

support_band_summary["fraction_test"] = (
    support_band_summary["n_test_molecules"]
    / len(tanimoto_support)
)

support_band_summary

,support_band,n_test_molecules,fraction_test
0,low_support_<0.50,63,0.203226
1,moderate_support_0.50_0.70,113,0.364516
2,high_support_0.70_0.85,113,0.364516
3,very_high_support_>=0.85,21,0.067742


**Near-Duplicate / Chemical-Support Result**

No exact molecule duplication was found across train and test partitions. Nearest-neighbor Morgan Tanimoto analysis showed mixed chemical support for the final scaffold-held-out test set.

Most test molecules were not extreme near-duplicates of training molecules. Only 21 of 310 test molecules (6.8%) had nearest-train Tanimoto similarity greater than or equal to 0.85.

However, 113 test molecules (36.5%) fell in the high-support range of 0.70–0.85, and the median nearest-train Tanimoto similarity was approximately 0.69. This indicates that the final test set is scaffold-disjoint and exact-duplicate-free, but not chemically isolated.

This result supports proceeding to final test evaluation, with the caveat that Notebook 11 final-test performance should be interpreted alongside nearest-train Tanimoto support bands.

### **Section 6: CV Fold-Level Robustness Review**

This section reviews the saved Notebook 09 scaffold-CV results for the selected ExtraTrees model.

The purpose is to determine whether the strong ExtraTrees benchmark result was robust across scaffold folds or whether the mean result was inflated by one unusually favorable fold.

This section does not refit models, tune hyperparameters, alter model selection, or compute final test metrics. It uses only saved Notebook 09 training-CV artifacts.

The review focuses on:

- fold-level average precision;
- fold-level early-retrieval performance;
- weak-fold behavior;
- fold-to-fold variability;
- whether ExtraTrees remained credible under unfavorable scaffold folds.

A strong mean score is not enough by itself. For the selected model to justify final test evaluation, performance should be supported by acceptable weak-fold behavior rather than depending on a single favorable validation fold.

In [13]:
# load saved Notebook 09 CV artifacts and inspect available fold-level fields.
BENCHMARK_DIR = paths.MODELING_DIR / "model_benchmarking"


cv_results = pd.read_parquet(BENCHMARK_DIR / "candidate_model_cv_results.parquet")
cv_summary = pd.read_parquet(BENCHMARK_DIR / "candidate_model_summary_flat.parquet")

print(cv_results.shape)
print(cv_summary.shape)

(20, 17)
(4, 34)


In [14]:
# ExtraTrees (our dominant training model from Notebook 09) fold-level robustness review.
# inspect whether selected-model CV performance was robust across folds.

SELECTED_MODEL = "extra_trees"

# filter cv_results to extra_trees model and select our fold_metric_cols
selected_fold_results = (
    cv_results
    .loc[cv_results["model_name"].eq(SELECTED_MODEL)]
    .sort_values("fold")
    .reset_index(drop=True)
)

selected_cv_summary = (
    cv_summary
    .loc[cv_summary["model_name"].eq(SELECTED_MODEL)]
    .reset_index(drop=True)
)

selected_fold_results

,model_name,fold,model_family,complexity_tier,n_fold_train,n_fold_valid,fit_time_seconds,score_time_seconds,total_fold_time_seconds,valid_active_rate,average_precision,roc_auc,balanced_accuracy,precision_at_5_percent,ef_at_5_percent,precision_at_10_percent,ef_at_10_percent
0,extra_trees,1,tree_ensemble,medium,1047,189,0.312213,0.055917,0.368130,0.788360,0.972418,0.903523,0.761577,1.0,1.268456,1.000000,1.268456
1,extra_trees,2,tree_ensemble,medium,1001,235,0.311244,0.056644,0.367888,0.825532,0.968303,0.889490,0.740634,1.0,1.211340,1.000000,1.211340
2,extra_trees,3,tree_ensemble,medium,1004,232,0.311550,0.055917,0.367467,0.866379,0.979213,0.892634,0.656315,1.0,1.154229,1.000000,1.154229
3,extra_trees,4,tree_ensemble,medium,1028,208,0.309005,0.053485,0.362489,0.716346,0.988714,0.970311,0.781424,1.0,1.395973,1.000000,1.395973
4,extra_trees,5,tree_ensemble,medium,864,372,0.302324,0.074995,0.377320,0.715054,0.892496,0.738332,0.502944,1.0,1.398496,0.973684,1.361694


In [15]:
selected_cv_summary

,model_name,valid_active_rate_mean,valid_active_rate_std,valid_active_rate_min,average_precision_mean,average_precision_std,average_precision_min,roc_auc_mean,roc_auc_std,roc_auc_min,...,ef_at_10_percent_min,fit_time_seconds_mean,fit_time_seconds_std,fit_time_seconds_min,score_time_seconds_mean,score_time_seconds_std,score_time_seconds_min,total_fold_time_seconds_mean,total_fold_time_seconds_std,total_fold_time_seconds_min
0,extra_trees,0.782334,0.066796,0.715054,0.960229,0.038644,0.892496,0.878858,0.08518,0.738332,...,1.154229,0.309267,0.004064,0.302324,0.059392,0.008804,0.053485,0.368659,0.005371,0.362489


**CV Fold-Level Robustness Result**

The saved Notebook 09 scaffold-CV artifacts show that the selected ExtraTrees model was not driven by a single favorable fold.

Across the five scaffold folds, average precision remained strong, with mean AP approximately 0.960 and weakest-fold AP approximately 0.892. Early-retrieval behavior was especially stable: Precision@5% was 1.00 in all five folds, and Precision@10% was 1.00 in four of five folds, with the weakest fold still approximately 0.974.

Fold 5 is the main caveat. It had the weakest broad-discrimination metrics, with ROC-AUC approximately 0.738 and balanced accuracy approximately 0.503. This suggests that fold 5 was not cleanly separated under threshold-style classification. However, BioPred’s primary objective is ranked early retrieval, and fold 5 still retained strong top-K retrieval.

Overall, the fold-level evidence supports the credibility of the selected ExtraTrees CV result for BioPred’s ranking objective. The result remains unusually strong, so Notebook 10 proceeds to a label-permutation negative control before allowing final scaffold-held-out test evaluation.

### **Section 7: Label-Permutation Negative Control**

This section performs a targeted negative-control experiment for the selected ExtraTrees model.

Notebook 09 produced unusually strong scaffold-CV early-retrieval performance. The preceding Notebook 10 audits found no direct feature leakage, no scaffold overlap, no exact molecule duplication across train/test, and no widespread near-duplicate domination. However, those checks do not fully answer whether the modeling workflow could still produce strong-looking results when the relationship between molecular structure and activity is destroyed.

To test that, this section shuffles the training labels and reruns the selected ExtraTrees workflow under the same scaffold-aware CV structure.

The purpose is to answer:

> If the activity labels are random, does the ExtraTrees workflow still produce strong AP and early-retrieval metrics?

Expected behavior:

- Real-label ExtraTrees performance should remain materially stronger than shuffled-label performance.
- Shuffled-label performance should collapse toward label-prevalence-driven or random-ranking behavior.
- If shuffled-label performance remains close to the real-label benchmark, the Notebook 09 result would require serious skepticism before final test evaluation.

This section is a negative control, not a tuning experiment.

Rules:

- Use training data only.
- Do not use final test labels.
- Do not alter the selected model family.
- Do not use shuffled-label results for model selection.
- Do not tune hyperparameters based on this experiment.
- Use the locked scaffold-aware CV grouping policy.

In [16]:
# setup cell for the augmented model run
# Section 7A: define selected ExtraTrees workflow objects.
# Recreate the selected Notebook 09 ExtraTrees preprocessing, estimator, and CV policy.

SELECTED_SPLIT_SEED = 33
PRIMARY_TARGET = "active_median_px_6_0"
SCAFFOLD_COL = "bemis_murcko_scaffold"

rdkit_cols = train_rdkit_cols
morgan_cols = train_morgan_cols

assert len(rdkit_cols) == 217
assert len(morgan_cols) == 2048

rdkit_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("rdkit", rdkit_preprocessor, rdkit_cols),
        ("morgan", "passthrough", morgan_cols),
    ],
    remainder="drop",
)

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SELECTED_SPLIT_SEED,
)

selected_extra_trees = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=SELECTED_SPLIT_SEED,
    n_jobs=-1,
    class_weight=None,
)


In [17]:
# Section 7B: selected-model scaffold-CV evaluator.
# provide one clean function for the label-permutation negative control.

def run_selected_model_scaffold_cv(
    *,
    X,
    y,
    cv_splits,
    estimator,
    preprocessor,
    model_name,
    label_condition,
):
    """
    Run scaffold-aware CV for the selected model workflow.

    This function is used for the label-permutation negative control.
    It does not touch the final scaffold-held-out test set.
    """
    fold_rows = []

    for fold_id, (train_idx, valid_idx) in enumerate(cv_splits, start=1):
        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        pipeline = Pipeline(
            steps=[
                ("preprocessor", clone(preprocessor)),
                ("model", clone(estimator)),
            ]
        )

        pipeline.fit(X_train_fold, y_train_fold)

        y_valid_score = pipeline.predict_proba(X_valid_fold)[:, 1]
        y_valid_pred = pipeline.predict(X_valid_fold)

        fold_rows.append(
            {
                "model_name": model_name,
                "label_condition": label_condition,
                "fold": fold_id,
                "n_fold_train": len(train_idx),
                "n_fold_valid": len(valid_idx),
                "valid_active_rate": float(y_valid_fold.mean()),
                "average_precision": average_precision_score(y_valid_fold, y_valid_score),
                "roc_auc": roc_auc_score(y_valid_fold, y_valid_score),
                "balanced_accuracy": balanced_accuracy_score(y_valid_fold, y_valid_pred),
                "precision_at_5_percent": precision_at_k_percent(
                    y_true=y_valid_fold,
                    y_score=y_valid_score,
                    k_percent=0.05,
                ),
                "ef_at_5_percent": enrichment_factor_at_k_percent(
                    y_true=y_valid_fold,
                    y_score=y_valid_score,
                    k_percent=0.05,
                ),
                "precision_at_10_percent": precision_at_k_percent(
                    y_true=y_valid_fold,
                    y_score=y_valid_score,
                    k_percent=0.10,
                ),
                "ef_at_10_percent": enrichment_factor_at_k_percent(
                    y_true=y_valid_fold,
                    y_score=y_valid_score,
                    k_percent=0.10,
                ),
            }
        )

    return pd.DataFrame(fold_rows)

In [18]:
# Section 7C: create shuffled training labels.
# preserve the training label balance while destroying the structure-activity relationship.

PERMUTATION_SEED = 733

y_train_primary = y_train[PRIMARY_TARGET]
groups_train = split_assignments_train[SCAFFOLD_COL]

assert X_train_raw.index.equals(y_train_primary.index)
assert X_train_raw.index.equals(groups_train.index)

rng = np.random.default_rng(PERMUTATION_SEED)

y_train_permuted = pd.Series(
    rng.permutation(y_train_primary.to_numpy()),
    index=y_train_primary.index,
    name=PRIMARY_TARGET,
)

permutation_check = pd.DataFrame(
    [
        {
            "label_condition": "real_labels",
            "active_rate": y_train_primary.mean(),
            "n_active": int(y_train_primary.sum()),
            "n_inactive": int((1 - y_train_primary).sum()),
        },
        {
            "label_condition": "permuted_labels",
            "active_rate": y_train_permuted.mean(),
            "n_active": int(y_train_permuted.sum()),
            "n_inactive": int((1 - y_train_permuted).sum()),
        },
    ]
)

permutation_check

,label_condition,active_rate,n_active,n_inactive
0,real_labels,0.77589,959,277
1,permuted_labels,0.77589,959,277


In [19]:
# create fixed scaffold-CV splits from the real training labels.
# these folds are reused for the permuted-label negative control.

fixed_cv_splits = list(
    cv.split(
        X_train_raw,
        y_train_primary,
        groups=groups_train,
    )
)

fixed_cv_split_summary = pd.DataFrame(
    [
        {
            "fold": fold_id,
            "n_fold_train": len(train_idx),
            "n_fold_valid": len(valid_idx),
            "valid_active_rate_real_labels": float(y_train_primary.iloc[valid_idx].mean()),
        }
        for fold_id, (train_idx, valid_idx) in enumerate(fixed_cv_splits, start=1)
    ]
)

fixed_cv_split_summary

,fold,n_fold_train,n_fold_valid,valid_active_rate_real_labels
0,1,1047,189,0.788360
1,2,1001,235,0.825532
2,3,1004,232,0.866379
3,4,1028,208,0.716346
4,5,864,372,0.715054


In [20]:
permuted_cv_results = run_selected_model_scaffold_cv(
    X=X_train_raw,
    y=y_train_permuted,
    cv_splits = fixed_cv_splits,
    estimator=selected_extra_trees,
    preprocessor=preprocessor,
    model_name="extra_trees",
    label_condition="permuted_labels",
)

permuted_cv_results

,model_name,label_condition,fold,n_fold_train,n_fold_valid,valid_active_rate,average_precision,roc_auc,balanced_accuracy,precision_at_5_percent,ef_at_5_percent,precision_at_10_percent,ef_at_10_percent
0,extra_trees,permuted_labels,1,1047,189,0.814815,0.789375,0.415399,0.426623,1.0,1.227273,1.0,1.227273
1,extra_trees,permuted_labels,2,1001,235,0.761702,0.793274,0.523943,0.512819,1.0,1.312849,1.0,1.312849
2,extra_trees,permuted_labels,3,1004,232,0.788793,0.768110,0.440337,0.496543,1.0,1.267760,1.0,1.267760
3,extra_trees,permuted_labels,4,1028,208,0.754808,0.746941,0.493069,0.490696,1.0,1.324841,1.0,1.324841
4,extra_trees,permuted_labels,5,864,372,0.768817,0.724532,0.439116,0.491828,1.0,1.300699,1.0,1.300699


**Label-Permutation CV Result**

The fixed-fold label-permutation negative control was run successfully. The validation fold sizes match the saved Notebook 09 scaffold-CV fold structure, confirming that the negative control changed the labels but preserved the same scaffold fold memberships.

Under permuted labels, the selected ExtraTrees workflow no longer showed strong broad discrimination. Average precision dropped into the approximate 0.72–0.79 range, ROC-AUC fell to approximately 0.42–0.52, and balanced accuracy stayed near random-threshold behavior.

Precision@5% and Precision@10% remained 1.00 across folds. This does not indicate a clean negative-control failure by itself because the dataset is highly active-heavy. With validation active rates around 0.75–0.81, top-K precision can remain saturated even when the molecule-label relationship has been destroyed. For this reason, AP, ROC-AUC, and balanced accuracy are treated as the more informative negative-control metrics in this section.

Overall, the permutation result supports the credibility of the real-label Notebook 09 ExtraTrees result: when structure-activity correspondence is broken, the workflow loses broad ranking and discrimination signal. The top-K saturation caveat should be carried into the final Notebook 10 audit decision.

### **Section 8: Final Pre-Test Audit Decision**

This section records the final Notebook 10 audit decision.

Notebook 10 was designed to determine whether the selected Notebook 09 ExtraTrees model result is credible enough to justify one-time final scaffold-held-out test evaluation in Notebook 11.

This section does not compute final test metrics, tune the selected model, change the selected model family, alter the train/test split, or modify the feature policy.

The decision options are:

- `pass`: proceed to Notebook 11 final scaffold-held-out test evaluation;
- `conditional_pass`: resolve specific concerns before final test evaluation;
- `fail`: do not touch final test metrics until the issue is corrected.

The decision is based on the full Notebook 10 audit record: artifact integrity, feature-policy compliance, scaffold split integrity, duplicate/near-duplicate risk, fold-level robustness, and label-permutation negative control behavior.

In [21]:
# record whether Notebook 10 supports proceeding to one-time final test evaluation.

audit_decision = {
    "notebook": "10_selected_model_validation_and_leakage_audit",
    "selected_model": "extra_trees",
    "selection_source": "09_candidate_model_benchmarking",
    "final_test_status": "not_used_in_notebook_10",
    "decision": "pass",
    "next_step": "proceed_to_notebook_11_final_scaffold_heldout_test_evaluation",
    "key_pass_findings": [
        "Notebook 09 selected ExtraTrees under scaffold-aware training-CV policy.",
        "Locked Notebook 07 train/test artifacts loaded correctly and remained row-aligned.",
        "No forbidden target, potency, diagnostic, identifier, SMILES, InChIKey, or scaffold columns were found in model feature matrices.",
        "Train/test scaffold overlap remained zero.",
        "No exact molecule identity overlap was found across train/test using available identifiers.",
        "Nearest-train Morgan Tanimoto support showed no widespread extreme near-duplicate domination; only a small minority of test molecules were very-high support.",
        "Saved Notebook 09 ExtraTrees fold-level results were not driven by a single favorable scaffold fold.",
        "Fixed-fold label-permutation negative control showed collapse in broad discrimination metrics when label correspondence was destroyed.",
    ],
    "caveats_for_notebook_11": [
        "Final test performance should be interpreted alongside nearest-train Tanimoto support bands.",
        "Fold 5 in Notebook 09 showed weaker broad-discrimination behavior despite strong early retrieval.",
        "Permutation-control Precision@K remained saturated because the dataset is active-heavy; AP, ROC-AUC, and balanced accuracy were more informative negative-control metrics.",
        "BioPred should avoid overclaiming broad chemical novelty generalization if final ranked performance concentrates in high-support or very-high-support molecules.",
    ],
}

audit_decision

{'notebook': '10_selected_model_validation_and_leakage_audit',
 'selected_model': 'extra_trees',
 'selection_source': '09_candidate_model_benchmarking',
 'final_test_status': 'not_used_in_notebook_10',
 'decision': 'pass',
 'next_step': 'proceed_to_notebook_11_final_scaffold_heldout_test_evaluation',
 'key_pass_findings': ['Notebook 09 selected ExtraTrees under scaffold-aware training-CV policy.',
  'Locked Notebook 07 train/test artifacts loaded correctly and remained row-aligned.',
  'No forbidden target, potency, diagnostic, identifier, SMILES, InChIKey, or scaffold columns were found in model feature matrices.',
  'Train/test scaffold overlap remained zero.',
  'No exact molecule identity overlap was found across train/test using available identifiers.',
  'Nearest-train Morgan Tanimoto support showed no widespread extreme near-duplicate domination; only a small minority of test molecules were very-high support.',
  'Saved Notebook 09 ExtraTrees fold-level results were not driven b

In [22]:
# save Notebook 10 audit decision artifact.

AUDIT_DIR = paths.MODELING_DIR / "selected_model_validation_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

audit_decision_path = AUDIT_DIR / "notebook_10_audit_decision.json"

with open(audit_decision_path, "w") as f:
    json.dump(audit_decision, f, indent=2)

audit_decision_path

PosixPath('/home/ryanm/projects/BioPred/data/processed/modeling/selected_model_validation_audit/notebook_10_audit_decision.json')

### **Final Audit Decision**

Notebook 10 supports a `pass` decision.

The selected ExtraTrees model from Notebook 09 passed the pre-final-test audit. The audit found no direct feature leakage, no train/test scaffold overlap, no exact molecule duplication across train/test, no widespread extreme near-duplicate domination, and no evidence that the strong CV result was driven only by one favorable fold.

The label-permutation negative control was the strongest stress test in this notebook. When activity labels were shuffled while preserving the fixed scaffold-CV fold memberships, the selected workflow lost broad discrimination signal: AP dropped toward prevalence-driven behavior, ROC-AUC fell to random/poor levels, and balanced accuracy remained near random-threshold behavior.

Two caveats carry forward. First, final test performance must be interpreted alongside nearest-train Tanimoto support bands. Second, top-K precision is less diagnostic under permutation because the dataset is highly active-heavy and Precision@K can saturate even when labels are randomized.

**Decision:** proceed to Notebook 11 for one-time final scaffold-held-out test evaluation.